In [8]:
import pandas as pd

# Load 4 NHANES datasets
demo = pd.read_sas('../data/DEMO_J.xpt')
bmx  = pd.read_sas('../data/BMX_J.xpt')
paq  = pd.read_sas('../data/PAQ_J.xpt')
pfq  = pd.read_sas('../data/PFQ_J.xpt')

print("DEMO:", demo.shape)
print("BMX: ", bmx.shape)
print("PAQ: ", paq.shape)
print("PFQ: ", pfq.shape)

DEMO: (9254, 46)
BMX:  (8704, 21)
PAQ:  (5856, 17)
PFQ:  (8421, 36)


In [9]:
# Select variables from each table
demo_sel = demo[['SEQN', 'RIDAGEYR', 'RIAGENDR']]
bmx_sel  = bmx[['SEQN', 'BMXBMI']]
paq_sel  = paq[['SEQN', 'PAD680', 'PAQ605']]
pfq_sel  = pfq[['SEQN', 'PFQ061D']]

# Merge 4 tables using SEQN
df = demo_sel.merge(bmx_sel, on='SEQN', how='inner')
df = df.merge(paq_sel, on='SEQN', how='inner')
df = df.merge(pfq_sel, on='SEQN', how='inner')

print("after merging:", df.shape)
print(df.head())

after merging: (5533, 7)
      SEQN  RIDAGEYR  RIAGENDR  BMXBMI  PAD680  PAQ605  PFQ061D
0  93705.0      66.0       2.0    31.7   300.0     2.0      1.0
1  93706.0      18.0       1.0    21.5   240.0     2.0      NaN
2  93708.0      66.0       2.0    23.7   120.0     2.0      3.0
3  93709.0      75.0       2.0    38.9   600.0     2.0      3.0
4  93711.0      56.0       1.0    21.3   420.0     2.0      NaN


In [10]:
# Filter age 20-65
df = df[(df['RIDAGEYR'] >= 20) & (df['RIDAGEYR'] <= 65)]

# Remove invalid sedentary time values
df = df[df['PAD680'] != 9999]

# Remove missing and invalid physical activity values
df = df[df['PAQ605'].notna()]
df = df[~df['PAQ605'].isin([7, 9])]

# Remove missing and invalid functional limitation values
df = df[df['PFQ061D'].notna()]
df = df[~df['PFQ061D'].isin([5, 7, 9])]

# Remove missing BMI
df = df[df['BMXBMI'].notna()]

# Convert sedentary time to hours and remove outliers
df['sitting_hours'] = df['PAD680'] / 60
df = df[(df['sitting_hours'] >= 0.5) & (df['sitting_hours'] <= 18)]

print("after cleaning:", df.shape)

after cleaning: (1374, 8)


In [11]:
# Binarize outcome variable: 1 = has difficulty, 0 = no difficulty
df['functional_limitation'] = (df['PFQ061D'] >= 2).astype(int)

# Binarize physical activity: 1 = active, 0 = not active
df['physically_active'] = (df['PAQ605'] == 1).astype(int)

print(df['functional_limitation'].value_counts())
print(df['physically_active'].value_counts())

functional_limitation
1    783
0    591
Name: count, dtype: int64
physically_active
0    1028
1     346
Name: count, dtype: int64
